In [1]:
clean_up = True # if True, remove all gams related files from working folder before starting
%run stdPackages.ipynb

The file _gams_py_gdb0.gdx is still active and was not deleted.
The file _gams_py_gdb1.gdx is still active and was not deleted.


*Load specific modules used here:*

In [2]:
os.chdir(d['py'])
import mWaste
from gmsPython import nestingTree
from valueShares import InitTree, nestedShares

# Production modules

Load data:

In [3]:
t0 = 2019
v = 'vGRS'
db = GpyDB(os.path.join(d['data'], f'{v}{t0}_db'), name = f'{v}{t0}_db', ws = d['work'])
ws = db.ws

For variables that are defined over $t$, but where we do not yet have an initial value for all $t$, extrapolate from data:

*Note: This forces extrapolation of all variables defined over $t$ - if it is important that some variables are not extrapolated, they should be removed from this statement.*

In [4]:
[symbol.__setattr__('vals', extrapolateUpper(symbol.vals, db('tE')[0])) for symbol in [db[k] for k in db.varDom('t')['t']]];

Subset data to only include production sectors, but keep a full copy:

In [5]:
db0 = db.copy()
sIdx = pd.Index(['Energy','Other'], name = 's')
AggDB.subsetDB(db, sIdx)

## 1. Nesting structure

In [6]:
nestInp = nestingTree.Tree('Inp', tree = db('nestProdInp').to_list(), f = 'CES_scaled')
nestOut = nestingTree.Tree('Out', tree = db('nestProdOut').to_list(), f= 'CET_scaled')
nestAgg = nestingTree.AggTree(name = 'P', trees = {t.name: t for t in [nestInp, nestOut]}, ws = ws)
nestAgg(namespace = {str(n)+'_input': n for n in db('n').union(db('nm_D'))})

*Add static user cost as initial value for prices on durables:*

In [7]:
db.aom(db('pD_dur'), name = 'pD', priority = 'first')

*Get initial values of $\mu$-parameters and intermediate goods:*

In [8]:
v = InitTree(nestAgg, ws = ws)
v.db['sigma'] = adj.rc_pd(db('sigma'), v.db('s'))
v.db['eta'] = adj.rc_pd(db('eta'), v.db('s'))
vals = v(dbIO = db)

*Add to main database:*

In [9]:
[db.aom(stdSort(adjMultiIndex.bc(vals[k], db('txE'))), name = k, priority = 'second') for k in ('pD','qD','pS','qS')];
db.aom(vals['mu'], name = 'mu', priority = 'second')
db.aom(gpy(1/vals['mu'], name = 'qNorm', type = 'par'), priority = 'second') # normalizing parameter in 

Initialize model instance:

In [10]:
[nestAgg.db.__setitem__(k, db(k)) for k in ('dWTn','dWTy','dWTyF')];
m = mWaste.DynamicNCES_emission_WG(nestAgg, exoP = db('d_pEqui'), exoQS = db('d_qSEqui'), partial = True, initFromGms = 'initFirmValueBlock')

Add data and initialize other stuff with default options:

In [11]:
m.initStuff(db = db)
AggDB.readSets(m.db)
m.db.mergeInternal()

Solve calibration:

In [12]:
solveDB = m.jSolve(10, state='C')

Add solution to main remove local init methods:

In [14]:
[m.db.__setitem__(k, solveDB[k]) for k in m.db.getTypes(['var']) if k in solveDB.symbols]; # use solution database
m.db.mergeInternal()
m.initFromGms = None # no longer use gams code to initialize variables

Test solve baseline version as well:

In [15]:
solB = m.solve(state = 'B')